# Logistic Regression with GridSearchCV: Group-Wise Validation

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Modelling / Evaluation

## Purpose

This notebook tunes a logistic regression for `Label_03` (**1 = Production, 0 = Changeover**) with `GridSearchCV` and a **group-wise** validation design, in which whole NC programs are held out. It uses all numeric signals except the target, instead of the 40-feature list of notebooks 02 and 03.

Measures against data leakage:

- All other label columns are excluded, and the **target itself is explicitly removed from the features** (checked by an assertion).
- Time and non-numeric columns are not used as features.
- Features with an absolute correlation above 0.3 with `Label_03` are dropped, to avoid trivial single-signal solutions.
- Imputation and scaling happen **inside** a scikit-learn `Pipeline`, so they are re-fitted in every cross-validation split.
- The train/test split and the cross-validation are **group-wise**: all rows of one NC program (column `Program_path`) are either in the training or in the test data.

## Workflow

Run notebook 01 first (it creates `normalized_data.csv`). Because notebook 01 removes text columns, the group labels are read separately from the original `data.csv`, whose rows are in the same order.

> **Data:** The dataset is public (Zenodo, licence CC BY 4.0) and is **not** included in this repository. See the project README for the download link and citation. This notebook reads `normalized_data.csv` (created by notebook 01) and the `Program_path` column of `data.csv`. Generated files are written to `../outputs/` and are not committed.

## Important limitations

- **Few groups.** The data contains only a handful of NC programs, so the test set consists of very few groups. The score depends strongly on which programs are held out. The notebook prints the class distribution of every group (Section 4); check it before trusting any score.
- **Feature filter on the whole dataset.** The correlation filter is computed on all rows, including test rows.
- **Earlier version.** An earlier version of this notebook accidentally kept `Label_03` among the features, which produced meaningless perfect scores. This is fixed. Results have to be re-generated by running the notebook.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt` (run from the project root). All paths are relative to this notebook, which is expected to live in the `notebooks/` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)

DATA_PATH = Path("../normalized_data.csv")   # created by notebook 01
RAW_DATA_PATH = Path("../data.csv")          # original dataset (only used for the group labels)
GROUP_COLUMN = "Program_path"
TARGET_COL = "Label_03"
CORR_THRESHOLD = 0.3
SEED = 42

## 2. Load the data and the group labels

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Run notebook 01 first or update DATA_PATH."
    )
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {RAW_DATA_PATH.resolve()}. "
        "Download data_v1_0_1.csv from https://doi.org/10.5281/zenodo.15735480 "
        "and save it as data.csv in the project root."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

if TARGET_COL not in df.columns:
    raise KeyError(f"Target column {TARGET_COL!r} not found in the data.")

# Group labels: the program path of every row, read from the original file
raw_columns = pd.read_csv(RAW_DATA_PATH, nrows=0).columns
matches = [c for c in raw_columns if c.lower().replace(" ", "_") == GROUP_COLUMN.lower()]
if not matches:
    raise KeyError(
        f"No column like {GROUP_COLUMN!r} found in {RAW_DATA_PATH}. "
        "Columns that mention 'program': "
        f"{[c for c in raw_columns if 'program' in c.lower()]}"
    )

groups = pd.read_csv(RAW_DATA_PATH, usecols=[matches[0]])[matches[0]].fillna("missing").astype(str)
if len(groups) != len(df):
    raise ValueError(
        "The original file and normalized_data.csv have different row counts "
        f"({len(groups)} vs {len(df)}). Re-run notebook 01 on the same data.csv."
    )
print(f"Number of groups ({matches[0]}): {groups.nunique()}")

## 3. Build features and target

Only numeric columns are candidates. The target and all other label columns are removed, as is `time`. Then every feature whose absolute correlation with the target exceeds `CORR_THRESHOLD` is dropped.

In [ ]:
# Numeric candidates (the target is still included at this point so it can be correlated)
num_df = df.select_dtypes(include=[np.number]).copy()
num_df[TARGET_COL] = df[TARGET_COL].astype(int)

# Columns that must never be features
label_cols = [c for c in df.columns if c.startswith("Label_")]   # includes the target
drop_cols = list(label_cols)
if "time" in num_df.columns:
    drop_cols.append("time")

# Absolute correlation with the target
corr_abs = num_df.corr(numeric_only=True)[TARGET_COL].abs().sort_values(ascending=False)
high_corr_feats = [c for c in corr_abs.index if c != TARGET_COL and corr_abs[c] > CORR_THRESHOLD]

print(f"Features with absolute correlation > {CORR_THRESHOLD} (dropped):")
for c in high_corr_feats:
    print(f"  {c}: {corr_abs[c]:.3f}")

drop_cols = list(dict.fromkeys(drop_cols + high_corr_feats))

X = num_df.drop(columns=drop_cols, errors="ignore")
y = num_df[TARGET_COL].astype(int).to_numpy()

# Safeguards against target leakage
assert TARGET_COL not in X.columns, "The target must not be part of the features."
assert not any(c.startswith("Label_") for c in X.columns), "No label column may be a feature."

print(f"\nFinal feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")

## 4. Group-wise train/test split

`GroupShuffleSplit` holds out about 25 % of the **groups**. The table first shows how many rows and what share of Production (class 1) each group contains. A test group that contains only one class makes the scores degenerate (for example a perfect score for trivial reasons), so read this table carefully.

In [ ]:
group_table = (
    pd.DataFrame({"group": groups.to_numpy(), "label": y})
    .groupby("group")["label"]
    .agg(rows="size", share_production="mean")
    .round(3)
)
display(group_table)

gss = GroupShuffleSplit(test_size=0.25, n_splits=1, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print(f"Train rows: {len(train_idx):,} in {groups_train.nunique()} groups")
print(f"Test rows:  {len(test_idx):,} in {groups_test.nunique()} groups")
print("Test groups:", sorted(groups_test.unique()))

single_class_test_groups = [
    g for g in groups_test.unique() if len(np.unique(y_test[groups_test.to_numpy() == g])) == 1
]
if single_class_test_groups:
    print("\nWarning: these test groups contain only one class:", single_class_test_groups)
if groups_train.nunique() < 4:
    raise ValueError("At least 4 training groups are needed for the 4-fold GroupKFold.")

## 5. Grid search with group-wise cross-validation

The pipeline imputes, standardizes and fits a logistic regression with `class_weight="balanced"`. `GridSearchCV` tries 4 values of `C` and 2 penalties (L1/L2) and selects by F1-score. `GroupKFold` makes sure that rows of one program are never in the training and validation part of the same split.

In [ ]:
pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear",
    )),
])

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__penalty": ["l1", "l2"],
}

cv_splits = list(GroupKFold(n_splits=4).split(X_train, y_train, groups=groups_train))

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=cv_splits,
    n_jobs=-1,
)

print("Running GridSearchCV with group-wise cross-validation...")
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print(f"Best cross-validated F1-score: {grid.best_score_:.4f}")

best_model = grid.best_estimator_

## 6. Evaluation on the held-out test programs

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

try:
    auc = roc_auc_score(y_test, y_proba)
except ValueError:      # happens if the test data contains only one class
    auc = None

print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Test precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Test recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Test F1-score:  {f1_score(y_test, y_pred, zero_division=0):.4f}")
if auc is not None:
    print(f"Test ROC-AUC:   {auc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion matrix: LR GridSearch (group-wise)")
plt.tight_layout()
plt.show()

## 7. Summary and conclusion

This notebook evaluates a tuned logistic regression on **unseen NC programs**, which is a stricter test than the random split of notebooks 02 and 03. Because the test set contains only very few programs, the result should be read together with the group table in Section 4 and with the cross-validated F1-score from Section 5.

No reference results are listed here: the numbers from the earlier version are invalid because the target was part of the features. Run the notebook on the downloaded dataset to obtain the real values, and compare the cross-validated F1-score with the test F1-score (a large gap points to an unrepresentative test split).